# ⛽ Gas Usage Distribution Analysis
## Dissertation Research - Blockchain Voting Performance

This notebook analyzes gas usage patterns in an Ethereum smart contract voting system.

### Key Research Questions:
1. What causes bimodal gas distribution?
2. How does cold vs warm storage access affect gas costs?
3. What is the cost impact of storage caching?

### Dataset:
- **File**: `performance-test-500voters-2026-02-03T12-06-19-226Z.csv`
- **Total Votes**: 500
- **Candidates**: 20
- **Voters**: 500

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Load data
data_path = Path('../report/performance-test-500voters-2026-02-03T12-06-19-226Z.csv')
df = pd.read_csv(data_path)

# Clean column names
df.columns = df.columns.str.strip()

# Display basic info
print("=" * 60)
print("📊 DATASET OVERVIEW")
print("=" * 60)
print(f"Total transactions: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## 1. Gas Distribution Analysis

In [ ]:
# Calculate exact cold/warm threshold from data
print("=" * 60)
print("🔍 CALCULATING COLD/WARM THRESHOLD")
print("=" * 60)

# Get unique gas values
unique_gas = df['Gas Used'].unique()
print(f"\nUnique gas values in dataset: {sorted(unique_gas)}")

# Find the gap between the two tiers
gas_sorted = sorted(df['Gas Used'].unique())
print(f"\nMin gas: {min(gas_sorted)}")
print(f"Max gas: {max(gas_sorted)}")

# Find the actual boundary (midpoint between tiers)
# Based on your data: Warm ~60,299-60,311, Cold ~77,399-77,411
# The boundary is somewhere between these

# First, let's VISUALIZE the gas distribution to see the actual gap
print("=" * 70)
print("🔍 STEP 1: VISUALIZE GAS DISTRIBUTION")
print("=" * 70)

# Get unique gas values
all_gas_values = sorted(df['Gas Used'].unique())
print(f"\n📊 Unique Gas Values Found ({len(all_gas_values)} unique values):")
print(f"   {all_gas_values}")

# Create visualization to see the gap
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: Histogram with ALL unique values marked
ax1 = axes[0]
df['Gas Used'].hist(bins=50, ax=ax1, color='steelblue', edgecolor='white', alpha=0.7)

# Mark each unique value
for gv in all_gas_values:
    ax1.axvline(gv, color='red', alpha=0.3, linewidth=1)

ax1.set_xlabel('Gas Used')
ax1.set_ylabel('Frequency')
ax1.set_title('Gas Distribution - Red Lines Show All Unique Gas Values')

# Bottom: Show gap between values
ax2 = axes[1]
gas_diffs = []
for i in range(len(all_gas_values) - 1):
    diff = all_gas_values[i+1] - all_gas_values[i]
    gas_diffs.append((all_gas_values[i], all_gas_values[i+1], diff))

# Sort by difference to find the largest gap
gas_diffs.sort(key=lambda x: x[2], reverse=True)

# Plot gaps
gap_positions = [f"{gd[0]}-{gd[1]}" for gd in gas_diffs[:15]]
gap_sizes = [gd[2] for gd in gas_diffs[:15]]
colors = ['red' if g > 1000 else 'green' for g in gap_sizes]

bars = ax2.barh(range(len(gap_sizes)), gap_sizes, color=colors, alpha=0.7)
ax2.set_yticks(range(len(gap_sizes)))
ax2.set_yticklabels(gap_positions)
ax2.set_xlabel('Gap Size (Gas Difference)')
ax2.set_title('Gaps Between Consecutive Gas Values (Red = Large Gap = Threshold Zone)')
ax2.axvline(1000, color='red', linestyle='--', alpha=0.5, label='Gap threshold')
ax2.legend()

plt.tight_layout()
plt.savefig('../report/fig0a_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 Gaps between consecutive gas values (sorted):")
for i, (g1, g2, diff) in enumerate(gas_diffs[:10]):
    marker = " ◄── LARGEST GAP (Threshold Zone!)" if i == 0 else ""
    print(f"   {g1:,} → {g2:,}: {diff:,} gas{marker}")

print("\n✅ Look at the visualization above to see the natural gap!")
print("   The largest gap is where the threshold should be.")

In [ ]:
# STEP 2: Calculate threshold based on the largest gap we just visualized
print("=" * 70)
print("🎯 STEP 2: CALCULATE THRESHOLD FROM GAP")
print("=" * 70)

# The largest gap is the natural boundary between cold and warm
largest_gap = gas_diffs[0]  # (value_before_gap, value_after_gap, gap_size)
max_warm = largest_gap[0]
min_cold = largest_gap[1]

# Threshold is the midpoint of the gap
cold_threshold = (max_warm + min_cold) // 2

print(f"\n✅ AUTOMATICALLY CALCULATED:")
print(f"   Max Warm Gas: {max_warm:,}")
print(f"   Min Cold Gas: {min_cold:,}")
print(f"   Gap Size: {largest_gap[2]:,} gas")
print(f"   ─────────────────────────────────")
print(f"   📌 THRESHOLD: {cold_threshold:,}")
print(f"   ─────────────────────────────────")
print(f"\n💡 This threshold is calculated from YOUR actual data!")
print(f"   Cold = Gas >= {cold_threshold:,}")
print(f"   Warm = Gas < {cold_threshold:,}")

In [ ]:
# Visualize the cold/warm threshold and gap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram with threshold and annotations
ax1 = axes[0]
df['Gas Used'].hist(bins=30, ax=ax1, color='steelblue', edgecolor='white', alpha=0.7)

# Add threshold line
ax1.axvline(cold_threshold, color='orange', linestyle='--', linewidth=3, 
            label=f'Threshold: {cold_threshold:,}')
ax1.axvline(max_warm, color='green', linestyle=':', linewidth=2, 
            label=f'Max Warm: {max_warm:,}')
ax1.axvline(min_cold, color='red', linestyle=':', linewidth=2, 
            label=f'Min Cold: {min_cold:,}')

# Shade the gap
ax1.axvspan(max_warm, min_cold, alpha=0.2, color='yellow', 
            label=f'Gap: {min_cold - max_warm:,} gas')

ax1.set_xlabel('Gas Used')
ax1.set_ylabel('Frequency')
ax1.set_title('Gas Distribution with Cold/Warm Threshold')
ax1.legend(loc='upper right')

# Right: Schematic diagram
ax2 = axes[1]
ax2.set_xlim(55000, 82000)
ax2.set_ylim(0, 2)

# Draw the two tiers as boxes
from matplotlib.patches import Rectangle
warm_box = Rectangle((55000, 0.5), max_warm - 55000, 0.8, 
                       facecolor='#51cf66', alpha=0.5, edgecolor='green', linewidth=2)
cold_box = Rectangle((min_cold, 0.5), 82000 - min_cold, 0.8, 
                       facecolor='#ff6b6b', alpha=0.5, edgecolor='red', linewidth=2)

ax2.add_patch(warm_box)
ax2.add_patch(cold_box)

# Add labels
ax2.text((55000 + max_warm)/2, 0.9, f'WARM\n{max_warm:,}', 
         ha='center', va='center', fontsize=12, fontweight='bold', color='green')
ax2.text((min_cold + 82000)/2, 0.9, f'COLD\n{min_cold:,}', 
         ha='center', va='center', fontsize=12, fontweight='bold', color='red')

# Add threshold marker
ax2.axvline(cold_threshold, color='orange', linestyle='--', linewidth=3)
ax2.text(cold_threshold, 1.6, f'THRESHOLD\n{cold_threshold:,}', 
         ha='center', va='center', fontsize=10, color='orange', fontweight='bold')

# Add gap annotation
ax2.annotate('', xy=(min_cold, 0.4), xytext=(max_warm, 0.4),
            arrowprops=dict(arrowstyle='<->', color='yellow', lw=2))
ax2.text((max_warm + min_cold)/2, 0.25, f'Gap: {min_cold - max_warm:,} gas', 
         ha='center', va='center', fontsize=11, color='darkgoldenrod')

ax2.set_title('Cold vs Warm Storage Access')
ax2.set_xlabel('Gas Used')
ax2.set_yticks([])
ax2.spines['left'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../report/fig0_threshold_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Threshold visualization saved: report/fig0_threshold_visualization.png")

In [ ]:
# Analyze gas usage distribution
print("=" * 60)
print("📈 GAS USAGE STATISTICS")
print("=" * 60)

gas_stats = df['Gas Used'].describe()
print(gas_stats)

# Identify cold vs warm storage - calculate threshold dynamically
# Find the gap between the two gas tiers
all_gas_values = sorted(df['Gas Used'].unique())
gap_idx = None
for i in range(len(all_gas_values) - 1):
    if all_gas_values[i+1] - all_gas_values[i] > 5000:  # Gap between tiers
        gap_idx = i
        break

# Threshold is midpoint of the gap
if gap_idx:
    cold_threshold = (all_gas_values[gap_idx] + all_gas_values[gap_idx + 1]) // 2
else:
    cold_threshold = 70000  # Default if no clear gap

print(f"🎯 Calculated cold/warm threshold: {cold_threshold:,}")

df['Storage Type'] = df['Gas Used'].apply(
    lambda x: 'Cold' if x >= cold_threshold else 'Warm'
)

# Count cold vs warm
cold_count = (df['Storage Type'] == 'Cold').sum()
warm_count = (df['Storage Type'] == 'Warm').sum()

print(f"\n🔴 Cold Storage Access: {cold_count} ({cold_count/len(df)*100:.1f}%)")
print(f"🔵 Warm Storage Access: {warm_count} ({warm_count/len(df)*100:.1f}%)")

# Calculate average gas for each type
avg_cold_gas = df[df['Storage Type'] == 'Cold']['Gas Used'].mean()
avg_warm_gas = df[df['Storage Type'] == 'Warm']['Gas Used'].mean()
gas_difference = avg_cold_gas - avg_warm_gas
gas_penalty_percent = (gas_difference / avg_warm_gas) * 100

print(f"\n📊 Average Gas by Type:")
print(f"   Cold Storage: {avg_cold_gas:,.0f} gas")
print(f"   Warm Storage: {avg_warm_gas:,.0f} gas")
print(f"   Difference: {gas_difference:,.0f} gas (~{gas_penalty_percent:.1f}% more)")

In [ ]:
# Create Figure 1: Gas Distribution Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram
ax1 = axes[0]
df['Gas Used'].hist(bins=30, ax=ax1, color='steelblue', edgecolor='white', alpha=0.7)
ax1.axvline(avg_cold_gas, color='red', linestyle='--', linewidth=2, label=f'Cold: {avg_cold_gas:,.0f}')
ax1.axvline(avg_warm_gas, color='green', linestyle='--', linewidth=2, label=f'Warm: {avg_warm_gas:,.0f}')
ax1.set_xlabel('Gas Used')
ax1.set_ylabel('Frequency')
ax1.set_title('Gas Usage Distribution (Bimodal)')
ax1.legend()

# Right: Pie chart
ax2 = axes[1]
colors = ['#ff6b6b', '#51cf66']
explode = (0.05, 0)
ax2.pie([cold_count, warm_count], labels=['Cold Storage', 'Warm Storage'], 
        autopct='%1.1f%%', colors=colors, explode=explode, startangle=90)
ax2.set_title('Storage Access Distribution')

plt.tight_layout()
plt.savefig('../report/fig1_gas_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Figure saved: report/fig1_gas_distribution.png")

## 2. Gas Distribution Table (For Dissertation)

In [ ]:
# Create frequency distribution table
print("=" * 70)
print("📊 GAS USAGE FREQUENCY DISTRIBUTION TABLE")
print("=" * 70)

# Create table data
table_data = {
    'Gas Level': ['Lower Gas', 'Higher Gas', 'Difference'],
    'Gas Values': ['~60,299 - 60,311', '~77,399 - 77,411', '~17,100 gas'],
    'Frequency': [f'{warm_count}', f'{cold_count}', '-'],
    'Percentage': [f'{warm_count/len(df)*100:.1f}%', f'{cold_count/len(df)*100:.1f}%', '~28% more expensive'],
    'Storage Type': ['🔵 Warm Access', '🔴 Cold Access', '⚠️ Cold penalty']
}

freq_df = pd.DataFrame(table_data)
print(freq_df.to_string(index=False))

# Save as CSV
freq_df.to_csv('../report/gas_frequency_table.csv', index=False)
print("\n✅ Table saved: report/gas_frequency_table.csv")

## 3. Analysis by Candidate

In [ ]:
# Analyze gas usage by candidate
candidate_stats = df.groupby('Candidate Name').agg({
    'Gas Used': ['count', 'mean', 'min', 'max'],
    'Storage Type': lambda x: (x == 'Cold').sum()
}).round(0)

candidate_stats.columns = ['Total Votes', 'Avg Gas', 'Min Gas', 'Max Gas', 'Cold Access Count']
candidate_stats['Warm Access Count'] = candidate_stats['Total Votes'] - candidate_stats['Cold Access Count']
candidate_stats['Cold %'] = (candidate_stats['Cold Access Count'] / candidate_stats['Total Votes'] * 100).round(1)
candidate_stats = candidate_stats.sort_values('Total Votes', ascending=False)

print("=" * 90)
print("📋 GAS USAGE BY CANDIDATE")
print("=" * 90)
print(candidate_stats.to_string())

# Save to CSV
candidate_stats.to_csv('../report/gas_by_candidate.csv')
print("\n✅ Table saved: report/gas_by_candidate.csv")

In [ ]:
# Create Figure 2: Gas Usage by Candidate
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Stacked bar chart
ax1 = axes[0]
candidates = candidate_stats.index.tolist()
cold_counts = candidate_stats['Cold Access Count'].values
warm_counts = candidate_stats['Warm Access Count'].values

x = np.arange(len(candidates))
width = 0.7

bars1 = ax1.bar(x, cold_counts, width, label='Cold Access', color='#ff6b6b')
bars2 = ax1.bar(x, warm_counts, width, bottom=cold_counts, label='Warm Access', color='#51cf66')

ax1.set_ylabel('Number of Votes')
ax1.set_xlabel('Candidate')
ax1.set_title('Cold vs Warm Storage Access by Candidate')
ax1.set_xticks(x)
ax1.set_xticklabels(candidates, rotation=45, ha='right')
ax1.legend()

# Right: Average gas per candidate
ax2 = axes[1]
avg_gas = candidate_stats['Avg Gas'].values
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(candidates)))
bars = ax2.barh(candidates, avg_gas, color=colors)
ax2.set_xlabel('Average Gas Used')
ax2.set_title('Average Gas Usage by Candidate')
ax2.axvline(avg_cold_gas, color='red', linestyle='--', alpha=0.7, label='Cold threshold')
ax2.axvline(avg_warm_gas, color='green', linestyle='--', alpha=0.7, label='Warm threshold')

plt.tight_layout()
plt.savefig('../report/fig2_gas_by_candidate.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Figure saved: report/fig2_gas_by_candidate.png")

## 4. Block-by-Block Analysis

In [ ]:
# Analyze gas usage by block
block_stats = df.groupby('Block Number').agg({
    'Gas Used': ['mean', 'count'],
    'Storage Type': lambda x: (x == 'Cold').sum()
}).reset_index()

block_stats.columns = ['Block Number', 'Avg Gas', 'Votes', 'Cold Count']
block_stats['Warm Count'] = block_stats['Votes'] - block_stats['Cold Count']

print("=" * 60)
print("📦 GAS USAGE BY BLOCK (First 20 Blocks)")
print("=" * 60)
print(block_stats.head(20).to_string(index=False))

# Find blocks with cold access
cold_blocks = block_stats[block_stats['Cold Count'] > 0]
print(f"\n🔴 Blocks with cold access: {len(cold_blocks)}")
print(f"🔵 Blocks with only warm access: {len(block_stats) - len(cold_blocks)}")

In [ ]:
# Create Figure 3: Block-by-block gas usage
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Average gas per block
ax1 = axes[0]
ax1.plot(block_stats['Block Number'], block_stats['Avg Gas'], 'b-', linewidth=1, alpha=0.7)
ax1.scatter(block_stats['Block Number'], block_stats['Avg Gas'], 
           c=block_stats['Cold Count'].apply(lambda x: 'red' if x > 0 else 'green'),
           s=50, alpha=0.8)

# Add horizontal lines for thresholds
ax1.axhline(avg_cold_gas, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Cold: {avg_cold_gas:,.0f}')
ax1.axhline(avg_warm_gas, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Warm: {avg_warm_gas:,.0f}')

ax1.set_xlabel('Block Number')
ax1.set_ylabel('Average Gas Used')
ax1.set_title('Gas Usage Pattern Over Blocks (Red = Cold Access Detected)')
ax1.legend()

# Bottom: Vote count per block with cold/warm breakdown
ax2 = axes[1]
ax2.bar(block_stats['Block Number'], block_stats['Cold Count'], 
       label='Cold Access', color='#ff6b6b', alpha=0.8)
ax2.bar(block_stats['Block Number'], block_stats['Warm Count'], 
       bottom=block_stats['Cold Count'], label='Warm Access', color='#51cf66', alpha=0.8)

ax2.set_xlabel('Block Number')
ax2.set_ylabel('Number of Votes')
ax2.set_title('Vote Distribution per Block')
ax2.legend()

plt.tight_layout()
plt.savefig('../report/fig3_gas_by_block.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Figure saved: report/fig3_gas_by_block.png")

## 5. Time Series Analysis (First Votes)

In [ ]:
# Analyze first votes to each candidate (demonstrates cold access)
df_sorted = df.sort_values('Voter Index').copy()

# Get first vote for each candidate
first_votes = df_sorted.groupby('Candidate Name').first().reset_index()

print("=" * 80)
print("🔴 FIRST VOTE TO EACH CANDIDATE (Demonstrates Cold Storage Access)")
print("=" * 80)
first_votes_display = first_votes[['Voter Index', 'Candidate Name', 'Gas Used', 'Block Number']]
first_votes_display.columns = ['Voter Index', 'Candidate', 'Gas Used', 'Block']
print(first_votes_display.to_string(index=False))

# Verify all first votes are cold
all_cold = (first_votes['Gas Used'] >= cold_threshold).all()
print(f"\n✅ All first votes are cold storage access: {all_cold}")
print(f"   Average gas for first votes: {first_votes['Gas Used'].mean():,.0f}")

In [ ]:
# Create Figure 4: Time series of first 100 transactions
fig, ax = plt.subplots(figsize=(14, 5))

# Get first 100 transactions sorted by voter index
first_100 = df_sorted.head(100)

# Plot gas usage
colors = first_100['Gas Used'].apply(lambda x: '#ff6b6b' if x >= cold_threshold else '#51cf66')
ax.scatter(range(100), first_100['Gas Used'], c=colors, s=50, alpha=0.7)

# Connect points
ax.plot(range(100), first_100['Gas Used'], 'gray', alpha=0.3, linewidth=1)

# Add threshold lines
ax.axhline(avg_cold_gas, color='red', linestyle='--', linewidth=2, label=f'Cold: {avg_cold_gas:,.0f}')
ax.axhline(avg_warm_gas, color='green', linestyle='--', linewidth=2, label=f'Warm: {avg_warm_gas:,.0f}')

ax.set_xlabel('Transaction Number (First 100)')
ax.set_ylabel('Gas Used')
ax.set_title('Gas Usage Over Time (First 100 Transactions)\n🔴 = Cold Access, 🔵 = Warm Access')
ax.legend()

plt.tight_layout()
plt.savefig('../report/fig4_gas_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Figure saved: report/fig4_gas_time_series.png")

## 6. Key Research Findings Summary

In [ ]:
# Generate comprehensive summary
print("=" * 80)
print("📝 KEY RESEARCH FINDINGS")
print("=" * 80)

findings = f"""
1. BIMODAL GAS DISTRIBUTION
   ├── Lower Gas Tier: ~{avg_warm_gas:,.0f} gas ({warm_count} votes, {warm_count/len(df)*100:.1f}%)
   ├── Higher Gas Tier: ~{avg_cold_gas:,.0f} gas ({cold_count} votes, {cold_count/len(df)*100:.1f}%)
   └── Difference: {gas_difference:,.0f} gas (~{gas_penalty_percent:.1f}% more for cold)

2. COLD vs WARM STORAGE ACCESS
   ├── Cold: First-time access to a storage slot (EVM cache miss)
   ├── Warm: Subsequent access to cached slot (EVM cache hit)
   └── Each candidate experiences exactly 1 cold access (first vote)

3. EXECUTION PATH DIFFERENCE
   ├── Different voters vote for different candidates
   ├── Each candidate's voteCount is a different storage slot
   └── First access to any slot = cold, subsequent = warm

4. COST IMPLICATIONS
   ├── Total gas without optimization: ~{avg_cold_gas * 500:,.0f} gas
   ├── Total gas with all warm: ~{avg_warm_gas * 500:,.0f} gas
   └── Potential savings: {gas_difference * 500:,.0f} gas (~{(1 - avg_warm_gas/avg_cold_gas) * 100:.1f}%)

5. EVM STORAGE CACHING
   ├── Ethereum maintains a "hot cache" of accessed storage slots
   ├── Cold access: {avg_cold_gas:,.0f} - {avg_warm_gas:,.0f} = {gas_difference:,.0f} extra gas
   └── This is a fundamental EVM optimization feature
"""

print(findings)

# Save findings to file
with open('../report/research_findings.txt', 'w') as f:
    f.write("GAS USAGE DISTRIBUTION ANALYSIS - RESEARCH FINDINGS\n")
    f.write("=" * 60 + "\n\n")
    f.write(findings)

print("✅ Findings saved: report/research_findings.txt")

## 7. Statistical Summary Table (For Dissertation)

In [ ]:
# Create comprehensive statistics table
stats_table = pd.DataFrame({
    'Metric': [
        'Total Votes',
        'Total Candidates',
        'Cold Storage Access Count',
        'Cold Storage Access %',
        'Warm Storage Access Count',
        'Warm Storage Access %',
        'Average Cold Gas',
        'Average Warm Gas',
        'Gas Difference',
        'Gas Penalty (%)',
        'Min Gas Used',
        'Max Gas Used',
        'Std Dev Gas'
    ],
    'Value': [
        len(df),
        df['Candidate Name'].nunique(),
        cold_count,
        f'{cold_count/len(df)*100:.1f}%',
        warm_count,
        f'{warm_count/len(df)*100:.1f}%',
        f'{avg_cold_gas:,.0f}',
        f'{avg_warm_gas:,.0f}',
        f'{gas_difference:,.0f}',
        f'{gas_penalty_percent:.1f}%',
        df['Gas Used'].min(),
        df['Gas Used'].max(),
        f'{df["Gas Used"].std():,.0f}'
    ]
})

print("=" * 60)
print("📊 STATISTICAL SUMMARY TABLE")
print("=" * 60)
print(stats_table.to_string(index=False))

# Save
stats_table.to_csv('../report/statistical_summary.csv', index=False)
print("\n✅ Table saved: report/statistical_summary.csv")

## 8. Visual Summary Figure (Publication Ready)

In [ ]:
# Create comprehensive publication-ready figure
fig = plt.figure(figsize=(16, 12))

# Create grid
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. Histogram (top left)
ax1 = fig.add_subplot(gs[0, 0])
df['Gas Used'].hist(bins=20, ax=ax1, color='steelblue', edgecolor='white', alpha=0.7)
ax1.axvline(avg_cold_gas, color='red', linestyle='--', linewidth=2)
ax1.axvline(avg_warm_gas, color='green', linestyle='--', linewidth=2)
ax1.set_xlabel('Gas Used')
ax1.set_ylabel('Frequency')
ax1.set_title('(a) Gas Distribution')

# 2. Pie chart (top middle)
ax2 = fig.add_subplot(gs[0, 1])
ax2.pie([cold_count, warm_count], labels=['Cold', 'Warm'], 
        autopct='%1.1f%%', colors=['#ff6b6b', '#51cf66'], startangle=90)
ax2.set_title('(b) Storage Access Type')

# 3. Statistics table (top right)
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')
table_data = [
    ['Metric', 'Value'],
    ['Total Votes', str(len(df))],
    ['Cold Access', f'{cold_count} ({cold_count/len(df)*100:.1f}%)'],
    ['Warm Access', f'{warm_count} ({warm_count/len(df)*100:.1f}%)'],
    ['Avg Cold Gas', f'{avg_cold_gas:,.0f}'],
    ['Avg Warm Gas', f'{avg_warm_gas:,.0f}'],
    ['Difference', f'{gas_difference:,.0f} ({gas_penalty_percent:.1f}%)']
]
table = ax3.table(cellText=table_data, loc='center', cellLoc='center',
                  colWidths=[0.5, 0.5])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax3.set_title('(c) Statistics')

# 4. First votes demonstration (middle left)
ax4 = fig.add_subplot(gs[1, 0])
first_vote_gas = first_votes['Gas Used'].values
ax4.bar(range(len(first_vote_gas)), first_vote_gas, color='#ff6b6b', alpha=0.8)
ax4.axhline(avg_cold_gas, color='red', linestyle='--', linewidth=1.5)
ax4.set_xlabel('Candidate (by ID)')
ax4.set_ylabel('Gas Used')
ax4.set_title('(d) First Vote per Candidate\n(All Cold Access)')

# 5. Gas by block (middle center & right)
ax5 = fig.add_subplot(gs[1, 1:])
ax5.plot(block_stats['Block Number'], block_stats['Avg Gas'], 'b-', linewidth=1, alpha=0.7)
ax5.scatter(block_stats['Block Number'], block_stats['Avg Gas'],
           c=block_stats['Cold Count'].apply(lambda x: 'red' if x > 0 else 'green'),
           s=60, alpha=0.8)
ax5.axhline(avg_cold_gas, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax5.axhline(avg_warm_gas, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
ax5.set_xlabel('Block Number')
ax5.set_ylabel('Average Gas')
ax5.set_title('(e) Gas Usage Over Blocks')

# 6. Time series (bottom)
ax6 = fig.add_subplot(gs[2, :])
colors = df_sorted.head(100)['Gas Used'].apply(lambda x: '#ff6b6b' if x >= cold_threshold else '#51cf66')
ax6.scatter(range(100), df_sorted.head(100)['Gas Used'], c=colors, s=40, alpha=0.7)
ax6.plot(range(100), df_sorted.head(100)['Gas Used'], 'gray', alpha=0.3, linewidth=1)
ax6.axhline(avg_cold_gas, color='red', linestyle='--', linewidth=2, label=f'Cold: {avg_cold_gas:,.0f}')
ax6.axhline(avg_warm_gas, color='green', linestyle='--', linewidth=2, label=f'Warm: {avg_warm_gas:,.0f}')
ax6.set_xlabel('Transaction Number')
ax6.set_ylabel('Gas Used')
ax6.set_title('(f) Gas Usage Time Series (First 100 Transactions)')
ax6.legend(loc='upper right')

# Main title
fig.suptitle('Gas Usage Distribution in Blockchain Voting System\n' + 
             f'500 Votes, 20 Candidates | Cold Penalty: {gas_difference:,.0f} gas (~{gas_penalty_percent:.1f}%)',
             fontsize=14, fontweight='bold', y=1.02)

plt.savefig('../report/fig5_comprehensive_summary.png', dpi=200, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

print("\n✅ Comprehensive figure saved: report/fig5_comprehensive_summary.png")
print("\n📁 All outputs saved to: report/")
print("   ├── fig1_gas_distribution.png")
print("   ├── fig2_gas_by_candidate.png")
print("   ├── fig3_gas_by_block.png")
print("   ├── fig4_gas_time_series.png")
print("   ├── fig5_comprehensive_summary.png")
print("   ├── gas_frequency_table.csv")
print("   ├── gas_by_candidate.csv")
print("   ├── statistical_summary.csv")
print("   └── research_findings.txt")